# Crop Mapping

Optional technical review notebook. The production CLI workflow lives in `../scripts/`.


# 02 | Freeze and validate the SPAM–FABLE crop mapping

**Historical SPAM crop-area harmonization · Production mapping stage**

This notebook consolidates the accepted mapping from native SPAM crop codes to the canonical **56 FABLE products** for the observed years **2000, 2005, 2010 and 2020**. It applies established one-to-one, many-to-one and country–year weighted allocations; it does **not** estimate new SPAM crop areas.

### Scientific conventions

- **SPAM supplies area magnitudes.** FAOSTAT data support the *composition of a broad-parent mapping* where the existing workflow accepted this evidence, not the absolute area magnitude.
- Mapping-v2 weights replace legacy weights for the **same country × year × SPAM parent**, not globally.
- The 2000 `ooil` parent uses the accepted seven-child mapping; groundnut and soybean are already explicit SPAM2000 crops.
- The 2000 `banp` and `swpy` mappings use accepted country composition, falling back to global-year composition only when documented support exists.
- The 2000 `othe` physical-area source total must remain available with an **unresolved irrigated/rainfed split**; this mapping notebook must not fabricate the split.
- Missing positive-area parents without supported mapping are reported, **not assigned arbitrary weights**.

**Scope boundary:** The later SPAM2000 missing-product backcasting, scientific-vs-operational completion and spatial reconstruction in the original development notebook are not executed here. Those require their own reconciled stage before a complete historical release can be claimed.

**Run order:** Notebook 01 → **Notebook 02** → SPAM2000 reconstruction → observed 5′ FABLE mapping → 2015 interpolation → independent validation.

## 01 | Paths and environment

Paths are configured from an environment variable where available, with the existing local FWat folder as the default. No desktop popup is required. This cell reads no large SPAM CSV files.

In [ ]:
# CELL 1 — PATHS, INPUT ASSETS, AND HELPERS
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path(os.environ.get("SPAM_PROJECT_ROOT", r"PROJECT_ROOT\WORK")).expanduser()
DATA_ROOT = PROJECT_ROOT / "Data"

FINAL_ROOT = DATA_ROOT / "SPAM_FABLE_FINAL"
NATIVE_DIR = FINAL_ROOT / "01_native_5arcmin"
OUTPUT_DIR = FINAL_ROOT / "02_mapping"
QC_DIR = OUTPUT_DIR / "qc"
for folder in [OUTPUT_DIR, QC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MAPPING_ROOT = DATA_ROOT / "mapping_audit"
MAPPING_MASTER_FILE = MAPPING_ROOT / "fable_crop_mapping_master_FINAL.csv"
LEGACY_WEIGHT_FILE = MAPPING_ROOT / "spam_country_year_weights_FINAL.csv"
V2_WEIGHT_FILE = (
    MAPPING_ROOT / "harmonized" / "global_master" / "script_08A_mapping"
    / "08A_SPAM_FABLE_mapping_v2_PRODUCTION_weights.csv"
)
NATIVE_COUNTRY_FILE = NATIVE_DIR / "01_SPAM_native_country_totals_QC.csv.gz"

# This is a FABLE schema reference only; no area values are imported from it.
SCRIPT07A_REFERENCE_FILE = (
    MAPPING_ROOT / "harmonized" / "global_master"
    / "fable_c_global_fable_product_irrigation_master_2000_2020.csv.gz"
)
PRODUCTLIST_FILE = DATA_ROOT / "productlistfinal.xlsx"

FAOSTAT_PANEL_FILE = (
    MAPPING_ROOT / "faostat"
    / "faostat_fable_target_years_common_country_panel.csv"
)

required = {
    "mapping master": MAPPING_MASTER_FILE,
    "legacy country-year weights": LEGACY_WEIGHT_FILE,
    "mapping-v2 production weights": V2_WEIGHT_FILE,
    "native SPAM country totals from Notebook 01": NATIVE_COUNTRY_FILE,
    "FAOSTAT mapping-composition panel": FAOSTAT_PANEL_FILE,
}
missing = [f"{name}: {path}" for name, path in required.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Required finalized mapping assets are missing:\n" + "\n".join(missing))

OBSERVED_YEARS = [2000, 2005, 2010, 2020]

FABLE_ALIASES = {
    "pulses other": "pulses_other",
    "pulses_other": "pulses_other",
    "citrus_othe": "citrus_other",
    "citrus_other": "citrus_other",
    # Mapping-v2 canonicalization accepted in 08A:
    "fiber_hard_other": "abaca",
    "fiber_soft_other": "jute",
}

def clean_string(series):
    return (
        series.astype("string").str.strip()
        .replace({"": pd.NA, "<NA>": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def normalize_fable_product(value):
    if pd.isna(value):
        return pd.NA
    text = str(value).strip()
    return FABLE_ALIASES.get(text, text)

def to_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return (
        series.astype("string").str.strip().str.lower()
        .isin(["true", "1", "yes", "y"])
    )

def sum_min_count(series):
    return pd.to_numeric(series, errors="coerce").sum(min_count=1)

print("SPAM mapping configuration ready")
print("Corrected Notebook 01 country asset:", NATIVE_COUNTRY_FILE)
print("Mapping master:", MAPPING_MASTER_FILE)
print("Legacy weights:", LEGACY_WEIGHT_FILE)
print("V2 weights:", V2_WEIGHT_FILE)
print("FAOSTAT composition panel:", FAOSTAT_PANEL_FILE)

## 02 | Freeze the canonical FABLE product universe

The FABLE product reference controls target names and output order. An existing Calculator schema reference is preferred; `productlistfinal.xlsx` is only a documented fallback, and its display ordering may differ. Exactly 56 unique products are required.

In [ ]:
# CELL 2 — FREEZE THE CANONICAL 56-PRODUCT FABLE REFERENCE

if SCRIPT07A_REFERENCE_FILE.exists():
    reference = pd.read_csv(SCRIPT07A_REFERENCE_FILE, compression="infer", low_memory=False)
    required_cols = ["calculator_order", "calculator_crop", "fable_product"]
    if not set(required_cols).issubset(reference.columns):
        raise KeyError(
            "The FABLE schema reference does not contain "
            "calculator_order, calculator_crop, and fable_product."
        )
    PRODUCT_REFERENCE = (
        reference[required_cols]
        .copy()
        .assign(
            fable_product=lambda x: x["fable_product"].map(normalize_fable_product),
            calculator_crop=lambda x: clean_string(x["calculator_crop"]),
            calculator_order=lambda x: pd.to_numeric(x["calculator_order"], errors="coerce"),
        )
        .dropna()
        .drop_duplicates()
        .sort_values("calculator_order")
        .reset_index(drop=True)
    )
else:
    # Fallback: authoritative productlistfinal.xlsx.
    # This recovers the 56 product universe, but not the exact Calculator display
    # label/order from the harmonized schema reference.
    if not PRODUCTLIST_FILE.exists():
        raise FileNotFoundError(
            "Neither the FABLE schema reference nor productlistfinal.xlsx was found."
        )
    raw = pd.read_excel(PRODUCTLIST_FILE, sheet_name=0, dtype=object)

    def norm_col(v):
        return re.sub(r"[^a-z0-9]+", "", str(v).replace("\xa0", " ").strip().casefold())

    fable_col = next(
        (c for c in raw.columns if norm_col(c) == "fable"),
        None,
    )
    if fable_col is None:
        raise KeyError("FABLE column not found in productlistfinal.xlsx.")

    products = (
        raw[fable_col].dropna().astype("string").str.strip()
        .map(normalize_fable_product).dropna().drop_duplicates().tolist()
    )
    PRODUCT_REFERENCE = pd.DataFrame({
        "calculator_order": range(1, len(products) + 1),
        "calculator_crop": products,
        "fable_product": products,
    })
    print(
        "WARNING: exact Calculator order/labels were not recovered from the old "
        "schema reference; productlist order is being used."
    )

if len(PRODUCT_REFERENCE) != 56 or PRODUCT_REFERENCE["fable_product"].nunique() != 56:
    display(PRODUCT_REFERENCE)
    raise ValueError(
        f"Expected exactly 56 canonical FABLE products; found "
        f"{len(PRODUCT_REFERENCE)} rows and "
        f"{PRODUCT_REFERENCE['fable_product'].nunique()} unique products."
    )

CANONICAL_PRODUCTS = PRODUCT_REFERENCE["fable_product"].tolist()
CANONICAL_PRODUCT_SET = set(CANONICAL_PRODUCTS)

PRODUCT_REFERENCE.to_csv(OUTPUT_DIR / "02_FABLE_56_product_reference.csv", index=False)

print("Canonical FABLE products:", len(CANONICAL_PRODUCTS))
display(PRODUCT_REFERENCE)

## 03 | Load source mapping, weights and corrected native-area audit

Read the frozen mapping tables, the FAOSTAT composition panel and **Notebook 01's corrected native country totals**. The `othe` source physical total is audited here without inventing an irrigation split. This step does not alter any native SPAM values.

In [ ]:
# CELL 3 — LOAD AND STANDARDIZE FINAL MAPPING ASSETS

MAPPING_MASTER = pd.read_csv(MAPPING_MASTER_FILE, low_memory=False)
MAPPING_MASTER["source_dataset"] = clean_string(MAPPING_MASTER["source_dataset"])
MAPPING_MASTER["source_product_code"] = clean_string(
    MAPPING_MASTER["source_product_code"]
).str.lower()
MAPPING_MASTER["source_version_or_year"] = pd.to_numeric(
    MAPPING_MASTER["source_version_or_year"], errors="coerce"
).astype("Int64")
MAPPING_MASTER["fable_product"] = MAPPING_MASTER["fable_product"].map(
    normalize_fable_product
)
if "include_in_water_area" in MAPPING_MASTER:
    MAPPING_MASTER["include_in_water_area"] = to_bool(
        MAPPING_MASTER["include_in_water_area"]
    )
else:
    MAPPING_MASTER["include_in_water_area"] = True

SPAM_MASTER = MAPPING_MASTER.loc[
    MAPPING_MASTER["source_dataset"].str.upper().eq("SPAM")
    & MAPPING_MASTER["source_version_or_year"].isin(OBSERVED_YEARS)
].copy()

def standardize_weights(df, is_v2=False):
    out = df.copy()
    out["country_iso3"] = clean_string(out["country_iso3"]).str.upper()

    if "year" in out.columns:
        year_series = out["year"]
    else:
        year_series = out["source_version_or_year"]
    out["year"] = pd.to_numeric(year_series, errors="coerce").astype("Int64")

    out["source_product_code"] = clean_string(out["source_product_code"]).str.lower()
    out["fable_product"] = out["fable_product"].map(normalize_fable_product)
    out["weight"] = pd.to_numeric(out["weight"], errors="coerce")

    for col in ["weight_basis", "weight_quality_flag"]:
        if col not in out.columns:
            out[col] = (
                "mapping_v2_production" if is_v2
                else "legacy_country_year_weight"
            )

    return out[
        [
            "country_iso3", "year", "source_product_code",
            "fable_product", "weight", "weight_basis", "weight_quality_flag"
        ]
    ].copy()

LEGACY = standardize_weights(pd.read_csv(LEGACY_WEIGHT_FILE, low_memory=False))
V2 = standardize_weights(pd.read_csv(V2_WEIGHT_FILE, low_memory=False), is_v2=True)

for label, table in [("legacy", LEGACY), ("v2", V2)]:
    invalid_target = table["fable_product"].notna() & ~table["fable_product"].isin(CANONICAL_PRODUCT_SET)
    if invalid_target.any():
        display(table.loc[invalid_target].head(100))
        raise ValueError(f"{label} weight table contains noncanonical FABLE targets.")

NATIVE_COUNTRY = pd.read_csv(NATIVE_COUNTRY_FILE, compression="infer", low_memory=False)
NATIVE_COUNTRY["country_iso3"] = clean_string(NATIVE_COUNTRY["country_iso3"]).str.upper()
NATIVE_COUNTRY["year"] = pd.to_numeric(NATIVE_COUNTRY["year"], errors="raise").astype(int)
NATIVE_COUNTRY["source_product_code"] = clean_string(
    NATIVE_COUNTRY["source_product_code"]
).str.lower()

FAOSTAT_PANEL = pd.read_csv(FAOSTAT_PANEL_FILE, low_memory=False)
FAOSTAT_PANEL["country_iso3"] = clean_string(FAOSTAT_PANEL["country_iso3"]).str.upper()
FAOSTAT_PANEL["year"] = pd.to_numeric(FAOSTAT_PANEL["year"], errors="coerce").astype("Int64")
FAOSTAT_PANEL["fable_product"] = FAOSTAT_PANEL["fable_product"].map(normalize_fable_product)
FAOSTAT_PANEL["harvested_total_ha"] = pd.to_numeric(
    FAOSTAT_PANEL["harvested_total_ha"], errors="coerce"
)

# Collapse duplicate FAOSTAT rows before using them only as composition evidence.
FAOSTAT_PANEL = (
    FAOSTAT_PANEL
    .groupby(["country_iso3", "year", "fable_product"], as_index=False)
    .agg(harvested_total_ha=("harvested_total_ha", sum_min_count))
)

print("SPAM mapping-master rows:", len(SPAM_MASTER))
print("Legacy weight rows:", len(LEGACY))
print("V2 production weight rows:", len(V2))
print("Native country-parent rows:", len(NATIVE_COUNTRY))

## 04 | Consolidate accepted country–year mapping weights

Preserve accepted mapping-v2 precedence. Rebuild only the documented 2000 broad-parent composition (`ooil`, and missing `banp`/`swpy` allocations). Country-year composition is preferred where complete; otherwise the accepted global-year composition fallback is used. Unsupported cases remain missing.

In [ ]:
# CELL 4 — CONSOLIDATE LEGACY + V2 WEIGHTS
# V2 replaces legacy only for the same country × year × parent.
# This preserves the accepted country-specific mapping-v2 decisions while
# retaining earlier validated weights where V2 does not replace them.

V2_KEY = ["country_iso3", "year", "source_product_code"]

v2_keys = V2[V2_KEY].drop_duplicates().assign(_v2=True)
LEGACY_RETAINED = (
    LEGACY.merge(v2_keys, on=V2_KEY, how="left")
    .loc[lambda x: x["_v2"].isna()]
    .drop(columns="_v2")
)

OPERATIONAL_WEIGHTS = pd.concat([LEGACY_RETAINED, V2], ignore_index=True)

# Correct SPAM-2000 OOIL globally using the child set validated in 08B.
# SPAM 2000 explicit crops already include groundnut and soybean, so they are
# excluded from the coarse OOIL allocation.
OOIL_2000_CHILDREN = [
    "coconut",
    "oilpalmfruit",
    "rapeseed",
    "sesame",
    "sunflower",
    "oilseed_other",
    "olive",
]

# 2000 BANP and SWPY are also coarse parents. Existing legacy weights are
# retained when available; only missing country-parent weights are patched.
BROAD_2000_PATCH_CHILDREN = {
    "banp": ["banana", "plantain"],
    "swpy": ["sweet_potato", "yams"],
}

def build_country_or_global_composition(country, year, parent, children):
    unknown = set(children) - CANONICAL_PRODUCT_SET
    if unknown:
        raise ValueError(f"Unknown FABLE children for {parent}: {sorted(unknown)}")

    country_table = (
        FAOSTAT_PANEL.loc[
            FAOSTAT_PANEL["country_iso3"].eq(country)
            & FAOSTAT_PANEL["year"].eq(year)
            & FAOSTAT_PANEL["fable_product"].isin(children),
            ["fable_product", "harvested_total_ha"],
        ]
        .drop_duplicates("fable_product")
        .set_index("fable_product")
        .reindex(children)
        .reset_index()
    )

    direct_complete = country_table["harvested_total_ha"].notna().all()
    direct_denom = country_table["harvested_total_ha"].sum(min_count=1)

    if direct_complete and pd.notna(direct_denom) and direct_denom > 0:
        selected = country_table
        basis = "FAOSTAT_country_year_observed_composition"
        quality = "country_year_observed_composition"
        denominator = direct_denom
    else:
        global_table = (
            FAOSTAT_PANEL.loc[
                FAOSTAT_PANEL["year"].eq(year)
                & FAOSTAT_PANEL["fable_product"].isin(children)
            ]
            .groupby("fable_product", as_index=False)
            .agg(harvested_total_ha=("harvested_total_ha", sum_min_count))
            .set_index("fable_product")
            .reindex(children)
            .reset_index()
        )
        global_complete = global_table["harvested_total_ha"].notna().all()
        global_denom = global_table["harvested_total_ha"].sum(min_count=1)

        if not global_complete or pd.isna(global_denom) or global_denom <= 0:
            return pd.DataFrame()

        selected = global_table
        basis = "FAOSTAT_global_year_composition_fallback"
        quality = "global_year_fallback"
        denominator = global_denom

    result = selected.copy()
    result["country_iso3"] = country
    result["year"] = year
    result["source_product_code"] = parent
    result["weight"] = result["harvested_total_ha"] / denominator
    result["weight_basis"] = basis
    result["weight_quality_flag"] = quality
    return result[
        [
            "country_iso3", "year", "source_product_code",
            "fable_product", "weight", "weight_basis", "weight_quality_flag",
        ]
    ]

# ---- Replace all 2000 OOIL rows with corrected seven-child composition -------
positive_parent = NATIVE_COUNTRY.loc[
    (NATIVE_COUNTRY["harvested_total_ha"].fillna(0) > 0)
    | (NATIVE_COUNTRY["physical_total_ha"].fillna(0) > 0)
].copy()

ooil_countries = (
    positive_parent.loc[
        positive_parent["year"].eq(2000)
        & positive_parent["source_product_code"].eq("ooil"),
        "country_iso3",
    ]
    .dropna().drop_duplicates().tolist()
)

ooil_parts = [
    build_country_or_global_composition(c, 2000, "ooil", OOIL_2000_CHILDREN)
    for c in ooil_countries
]
OOIL_2000_CORRECTED = (
    pd.concat([x for x in ooil_parts if not x.empty], ignore_index=True)
    if any(not x.empty for x in ooil_parts)
    else pd.DataFrame(columns=OPERATIONAL_WEIGHTS.columns)
)

OPERATIONAL_WEIGHTS = OPERATIONAL_WEIGHTS.loc[
    ~(
        OPERATIONAL_WEIGHTS["year"].eq(2000)
        & OPERATIONAL_WEIGHTS["source_product_code"].eq("ooil")
    )
].copy()
OPERATIONAL_WEIGHTS = pd.concat(
    [OPERATIONAL_WEIGHTS, OOIL_2000_CORRECTED],
    ignore_index=True,
)

# ---- Patch missing BANP / SWPY country weights only --------------------------
for parent, children in BROAD_2000_PATCH_CHILDREN.items():
    countries = (
        positive_parent.loc[
            positive_parent["year"].eq(2000)
            & positive_parent["source_product_code"].eq(parent),
            "country_iso3",
        ]
        .dropna().drop_duplicates().tolist()
    )

    covered = set(
        OPERATIONAL_WEIGHTS.loc[
            OPERATIONAL_WEIGHTS["year"].eq(2000)
            & OPERATIONAL_WEIGHTS["source_product_code"].eq(parent),
            "country_iso3",
        ].dropna()
    )

    missing_countries = [c for c in countries if c not in covered]
    patches = [
        build_country_or_global_composition(c, 2000, parent, children)
        for c in missing_countries
    ]
    patches = [x for x in patches if not x.empty]
    if patches:
        OPERATIONAL_WEIGHTS = pd.concat(
            [OPERATIONAL_WEIGHTS, *patches],
            ignore_index=True,
        )

OPERATIONAL_WEIGHTS = (
    OPERATIONAL_WEIGHTS
    .dropna(subset=["country_iso3", "year", "source_product_code", "fable_product", "weight"])
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Operational weight rows after consolidation:", len(OPERATIONAL_WEIGHTS))

## 05 | Freeze direct mapping edges

Exclude direct mapping edges for parent-year groups that already have an accepted weighted mapping; otherwise the same SPAM parent could be allocated twice. Reject invalid weights and duplicate edges.

In [ ]:
# CELL 5 — BUILD DIRECT EDGES, REMOVE PARENTS REPLACED BY WEIGHTED MAPPING

mapping_type = clean_string(SPAM_MASTER["mapping_type"]).str.lower()

DIRECT_EDGES = SPAM_MASTER.loc[
    SPAM_MASTER["include_in_water_area"]
    & SPAM_MASTER["fable_product"].notna()
    & ~mapping_type.eq("one_to_many_weighted")
    & SPAM_MASTER["fable_product"].isin(CANONICAL_PRODUCT_SET),
    [
        "source_version_or_year",
        "source_product_code",
        "fable_product",
        "mapping_type",
        "weight",
        "weight_basis",
    ],
].rename(columns={"source_version_or_year": "year"}).copy()

DIRECT_EDGES["year"] = pd.to_numeric(DIRECT_EDGES["year"], errors="raise").astype(int)
DIRECT_EDGES["weight"] = pd.to_numeric(DIRECT_EDGES["weight"], errors="coerce")

# Any parent-year that is represented by the accepted country-year weighted
# mapping must not simultaneously keep an old fixed/direct edge.
weighted_parent_years = (
    OPERATIONAL_WEIGHTS[["year", "source_product_code"]]
    .drop_duplicates()
    .assign(_weighted=True)
)

DIRECT_EDGES = (
    DIRECT_EDGES
    .merge(weighted_parent_years, on=["year", "source_product_code"], how="left")
    .loc[lambda x: x["_weighted"].isna()]
    .drop(columns="_weighted")
)

bad_direct = (
    DIRECT_EDGES["weight"].isna()
    | DIRECT_EDGES["weight"].lt(0)
    | DIRECT_EDGES["weight"].gt(1)
)
if bad_direct.any():
    display(DIRECT_EDGES.loc[bad_direct])
    raise ValueError("Invalid direct mapping weights.")

duplicate_direct = DIRECT_EDGES.duplicated(
    ["year", "source_product_code", "fable_product"], keep=False
)
if duplicate_direct.any():
    display(DIRECT_EDGES.loc[duplicate_direct])
    raise ValueError("Duplicate direct mapping edges.")

print("Direct operational edges:", len(DIRECT_EDGES))
display(DIRECT_EDGES.head(30))

## 06 | Conservation and positive-parent coverage

**Blocking test:** country–year–parent weighted allocations must sum to one within numerical tolerance. **Review output:** native parents with positive harvested or physical area but no accepted mapping remain visible. A source-total-only physical-area record such as SPAM2000 `othe` does not justify a constructed IR/RF split.

In [ ]:
# CELL 6 — STRICT WEIGHT CONSERVATION AND NATIVE-PARENT COVERAGE AUDIT

WEIGHT_QA = (
    OPERATIONAL_WEIGHTS
    .groupby(["country_iso3", "year", "source_product_code"], as_index=False)
    .agg(
        n_children=("fable_product", "nunique"),
        weight_sum=("weight", "sum"),
    )
)
WEIGHT_QA["weight_sum_error"] = (WEIGHT_QA["weight_sum"] - 1.0).abs()
bad_weights = WEIGHT_QA.loc[WEIGHT_QA["weight_sum_error"].gt(1e-10)]

if not bad_weights.empty:
    display(bad_weights.head(100))
    raise ValueError("Operational weighted mapping does not conserve parent area.")

# Country-parent inventory from the native SPAM 5′ dataset.
PARENT_INVENTORY = (
    NATIVE_COUNTRY
    .groupby(["country_iso3", "year", "source_product_code"], as_index=False)
    .agg(
        harvested_total_ha=("harvested_total_ha", sum_min_count),
        physical_total_ha=("physical_total_ha", sum_min_count),
    )
)
PARENT_INVENTORY["positive_parent_area"] = (
    PARENT_INVENTORY["harvested_total_ha"].fillna(0).gt(0)
    | PARENT_INVENTORY["physical_total_ha"].fillna(0).gt(0)
)

direct_keys = (
    DIRECT_EDGES[["year", "source_product_code"]]
    .drop_duplicates()
    .assign(direct_mapping_available=True)
)
weighted_keys = (
    OPERATIONAL_WEIGHTS[["country_iso3", "year", "source_product_code"]]
    .drop_duplicates()
    .assign(weighted_mapping_available=True)
)

PARENT_COVERAGE = (
    PARENT_INVENTORY
    .merge(direct_keys, on=["year", "source_product_code"], how="left")
    .merge(weighted_keys, on=["country_iso3", "year", "source_product_code"], how="left")
)
PARENT_COVERAGE["direct_mapping_available"] = (
    PARENT_COVERAGE["direct_mapping_available"].fillna(False)
)
PARENT_COVERAGE["weighted_mapping_available"] = (
    PARENT_COVERAGE["weighted_mapping_available"].fillna(False)
)
PARENT_COVERAGE["mapping_available"] = (
    PARENT_COVERAGE["direct_mapping_available"]
    | PARENT_COVERAGE["weighted_mapping_available"]
)

UNRESOLVED_POSITIVE_PARENTS = PARENT_COVERAGE.loc[
    PARENT_COVERAGE["positive_parent_area"]
    & ~PARENT_COVERAGE["mapping_available"]
].copy()

# Known unresolved broad residuals are NOT silently forced into FABLE products.
# They remain explicit so the final release can distinguish mapping evidence
# from unsupported assumptions.
print("Positive native country-parent groups without an accepted mapping:",
      len(UNRESOLVED_POSITIVE_PARENTS))
if not UNRESOLVED_POSITIVE_PARENTS.empty:
    display(
        UNRESOLVED_POSITIVE_PARENTS[
            ["country_iso3", "year", "source_product_code",
             "harvested_total_ha", "physical_total_ha"]
        ].head(100)
    )

WEIGHT_QA.to_csv(QC_DIR / "02_operational_weight_QA.csv", index=False)
PARENT_COVERAGE.to_csv(
    QC_DIR / "02_native_parent_mapping_coverage.csv.gz",
    index=False,
    compression="gzip",
)
UNRESOLVED_POSITIVE_PARENTS.to_csv(
    QC_DIR / "02_unresolved_positive_native_parents.csv.gz",
    index=False,
    compression="gzip",
)

## 07 | Verify the 2000 OTHE handoff and export the mapping assets

This is a *mapping-stage* export, not the final 56-product historical SPAM release. A separate SPAM2000 completion stage must reconcile any missing FABLE products and the previously approved MIRCA corrections before publication.

In [ ]:
# 07 | Corrected OTHE handoff and mapping-stage exports

from hashlib import sha256
import json

required_native_cols = {
    "country_iso3", "year", "source_product_code", "physical_total_ha",
    "physical_irrigated_ha", "physical_rainfed_ha"
}
missing_native_cols = required_native_cols - set(NATIVE_COUNTRY.columns)
if missing_native_cols:
    raise KeyError(f"Notebook 01 country file lacks required fields: {sorted(missing_native_cols)}")

othe = NATIVE_COUNTRY.loc[
    NATIVE_COUNTRY["year"].eq(2000)
    & NATIVE_COUNTRY["source_product_code"].eq("othe")
].copy()
if othe.empty:
    raise ValueError("Corrected SPAM2000 OTHE country totals are missing")

source_total = pd.to_numeric(othe["physical_total_ha"], errors="coerce").sum(min_count=1)
source_expected = 559_861_492.7  # Prior Notebook 01 source-total audit (ha)
abs_difference = abs(source_total - source_expected)
# Numerical comparison only; the source-total reference is not a replacement value.
if not np.isclose(source_total, source_expected, rtol=1e-10, atol=0.1):
    raise ValueError(
        f"SPAM2000 OTHE source total differs from validated Notebook 01: "
        f"{source_total:,.3f} vs {source_expected:,.3f} ha"
    )
if othe["physical_irrigated_ha"].notna().any() or othe["physical_rainfed_ha"].notna().any():
    raise ValueError("SPAM2000 OTHE has an invented IR/RF physical-area split")

# This stage exports mapping assets only. Never overwrite the reconstruction file.
DIRECT_EDGES.to_csv(OUTPUT_DIR / "02_SPAM_FABLE_direct_edges.csv", index=False)
OPERATIONAL_WEIGHTS.to_csv(
    OUTPUT_DIR / "02_SPAM_FABLE_operational_country_year_weights.csv.gz",
    index=False, compression="gzip"
)

mapping_summary = pd.DataFrame([
    {"check": "canonical_products", "value": len(CANONICAL_PRODUCTS), "status": "PASS"},
    {"check": "direct_edges", "value": len(DIRECT_EDGES), "status": "PASS"},
    {"check": "weighted_edges", "value": len(OPERATIONAL_WEIGHTS), "status": "PASS"},
    {"check": "unresolved_positive_native_parents", "value": len(UNRESOLVED_POSITIVE_PARENTS),
     "status": "REVIEW" if len(UNRESOLVED_POSITIVE_PARENTS) else "PASS"},
    {"check": "2000_OTHE_physical_total_ha", "value": source_total, "status": "PASS"},
    {"check": "2000_OTHE_IR_RF_split", "value": "unavailable in source", "status": "DOCUMENTED"},
])
mapping_summary.to_csv(QC_DIR / "02_mapping_stage_summary.csv", index=False)

input_assets = [MAPPING_MASTER_FILE, LEGACY_WEIGHT_FILE, V2_WEIGHT_FILE,
                NATIVE_COUNTRY_FILE, FAOSTAT_PANEL_FILE]
input_manifest = []
for path in input_assets:
    h = sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            h.update(chunk)
    input_manifest.append({"path": str(path), "sha256": h.hexdigest()})
with (OUTPUT_DIR / "02_mapping_input_manifest.json").open("w", encoding="utf-8") as stream:
    json.dump(input_manifest, stream, indent=2)

print("Mapping-stage QC and exports completed.")
print(f"2000 OTHE source physical total: {source_total:,.3f} ha")
print(f"Unresolved positive native parents: {len(UNRESOLVED_POSITIVE_PARENTS):,}")
print("STATUS: MAPPING_STAGE_READY_FOR_RECONSTRUCTION_REVIEW")
display(mapping_summary)

## Interpretation and next stage

Passing this notebook demonstrates that the **mapping stage** is reproducible with its required frozen source assets and that the accepted weight-conservation rules were satisfied. It does **not** mean that the full five-year, 56-product SPAM dataset is complete. The next dedicated stage must document SPAM2000 missing-product reconstruction, correction precedence, provenance and any unresolved country–product records.

### Primary outputs

- `02_FABLE_56_product_reference.csv`
- `02_SPAM_FABLE_direct_edges.csv`
- `02_SPAM_FABLE_operational_country_year_weights.csv.gz`
- `qc/02_operational_weight_QA.csv`
- `qc/02_native_parent_mapping_coverage.csv.gz`
- `qc/02_unresolved_positive_native_parents.csv.gz`
- `qc/02_mapping_stage_summary.csv`
- `02_mapping_input_manifest.json`

**Reproducibility note:** The original development notebook and its extensive SPAM2000 reconstruction experiments should be retained in the research archive, rather than silently presented as part of this frozen mapping stage.